# Bài 29: Xử lý dữ liệu lớn với MPC

Khi vùng nghiên cứu lớn (tỉnh, quốc gia) hoặc cần xử lý nhiều năm dữ liệu, RAM và thời gian xử lý tuần tự trở thành nút thắt cổ chai. Trong trường hợp như vậy, ta có thể chia nhỏ vùng nghiên cứu và dùng tính toán song song.

## 29.1. Mục tiêu học tập

Sau khi hoàn thành bài này, bạn có thể:

- Dùng `mercantile` để chia AOI thành tiles (XYZ), kiểm soát kích thước từng tile
- Lọc STAC items theo tile bbox để chỉ load dữ liệu cần thiết
- Xử lý nhiều tiles song song với `dask.delayed` và `LocalCluster`
- Lưu kết quả sử dụng GeoTIFF với nén dữ liệu

In [11]:
# Nếu chưa cài, hãy chạy lệnh sau để cài các thư viện cần thiết trước khi chạy notebook này:
import pystac_client
import planetary_computer as pc
from odc.stac import stac_load

import mercantile                       # XYZ tile utilities
import dask
import numpy as np
from functools import partial
import geopandas as gpd
from shapely.geometry import box

## 29.2. Đọc dữ liệu Sentinel-2

Trong phần này, chúng ta xem xét hai cách đọc dữ liệu Sentinel-2 từ Planetary Computer: đọc trực tiếp toàn bộ AOI hoặc chia nhỏ AOI thành các tile. Khi không dùng `mercantile`, toàn bộ vùng dữ liệu được tải cùng lúc, phù hợp với AOI nhỏ nhưng dễ gây tốn bộ nhớ. Khi dùng `mercantile`, AOI được chia thành các tile nhỏ và xử lý lần lượt, giúp giảm tải dữ liệu và tối ưu hiệu năng. Cách tiếp cận theo tile thường được sử dụng cho các bài toán quy mô lớn trên dữ liệu vệ tinh.

In [12]:
# Kết nối đến STAC API của Microsoft Planetary Computer.
mpc_url = "https://planetarycomputer.microsoft.com/api/stac/v1"

catalog = pystac_client.Client.open(
    mpc_url,
    modifier = pc.sign_inplace  # tự động ký URL asset
)

### 29.2.1. Đọc toàn bộ AOI

In [13]:
# Ví dụ đọc dữ dữ liệu Việt Nam 
vn = gpd.read_file('https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_VNM_0.json')
# lấy bounding box của Việt Nam để tải dữ liệu vệ tinh cho khu vực này
bbox = vn.total_bounds  # (xmin, ymin, xmax, ymax)

In [ ]:
# Tìm kiếm các item trong catalog của MPC cho toàn bộ Việt Nam
search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime="2023-01-01/2023-12-31",
    query={"eo:cloud_cover": {"lt": 10}}  # Chỉ lấy ảnh có độ che phủ mây < 10%
)

items = list(search.item_collection())
print(f"✅ Tìm thấy {len(items)} item Sentinel-2 phù hợp với tiêu chí tìm kiếm") 

In [ ]:
# Đọc dữ liệu từ các item vào xarray dataset với stac_load
data = stac_load(
    items = items,
    bands = ["B04", "B08"],  # Chỉ lấy 2 băng phổ đỏ và hồng ngoại gần để tính NDVI sau này
    chunks = {"x": 1024, "y": 1024}, # Chia nhỏ dữ liệu thành các chunk để xử lý với Dask
    resolution=10,
    patch_url=pc.sign,
    bbox=bbox
) # Dữ liệu hiện tại có shape (y: 166648, x: 80699, time: 238) trong năm 2023 với ảnh dưới 10% mây. Mặc dù dữ liệu đang ở trạng thái lazy (dask array), nhưng nếu chúng ta kích hoạt tính toán `compute` nó sẽ tải toàn bộ dữ liệu vào bộ nhớ, điều này sẽ gây ra lỗi thiếu bộ nhớ (out of memory) vì kích thước dữ liệu quá lớn để xử lý trên một máy tính cá nhân.

### 29.2.2. Đọc dữ liệu theo tile

Giả sử bạn muốn tính toán NDVI từ ảnh Sentinel-2 cho toàn bộ Việt Nam. Với diện tích rất lớn và khối lượng dữ liệu lên tới hàng chục hoặc hàng trăm GB, việc tải và xử lý toàn bộ ảnh trong một lần thường không khả thi do giới hạn bộ nhớ và thời gian tính toán. Một giải pháp phổ biến là chia AOI thành nhiều ô nhỏ (tiles) rồi xử lý từng tile độc lập.

In [ ]:
vn = gpd.read_file('https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_VNM_0.json')
# Tạo tiles XYZ để cover toàn bộ Việt Nam
bbox = vn.total_bounds  # (xmin, ymin, xmax, ymax)
zoom = 13 # Chọn zoom level phù hợp với kích thước vùng cần xử lý
tile_generator = partial(mercantile.tiles, zooms=zoom)
tiles = list(tile_generator(*bbox))
bbox_tiles = [box(*mercantile.bounds(tile)).bounds for tile in tiles]
print(f"✅ Đã tạo {len(tiles)} tiles ở zoom {zoom} để cover Việt Nam")

In [ ]:
# Ta chọn 1 bbox trong bbox_tiles để tải một phần dữ liệu nhỏ hơn để có thể xử lý trên máy cá nhân
tile_bbox = bbox_tiles[0]  # Lấy bbox của tile đầu tiên

# Tìm kiếm các item trong catalog của MPC cho tile_bbox
search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=tile_bbox,
    datetime="2023-01-01/2023-01-31",
    query={"eo:cloud_cover": {"lt": 2}}  # Chỉ lấy ảnh có độ che phủ mây < 5%
)

items = list(search.item_collection())
print(f"✅ Tìm thấy {len(items)} item Sentinel-2 phù hợp với tiêu chí tìm kiếm") 

In [ ]:
# Đọc dữ liệu từ các item vào xarray dataset với stac_load
data = stac_load(
    items = items,
    bands = ["B04", "B08"],  # Chỉ lấy 2 băng phổ đỏ và hồng ngoại gần để tính NDVI sau này
    chunks = {"x": 512, "y": 512}, # Chia nhỏ dữ liệu thành các chunk để xử lý với Dask
    resolution=10,
    patch_url=pc.sign,
    bbox=tile_bbox
) # Hiện tại dữ liệu vẫn ở dạng lazy (dask array), chưa được tính toán. 
print(f"✅ Đã tải dữ liệu cho tile đầu tiên với shape: {data['B04'].shape} và dtype: {data['B04'].dtype}")

In [ ]:
# Tính toán NDVI từ 2 băng phổ đỏ (B04) và hồng ngoại gần (B08). Công thức tính NDVI là: NDVI = (NIR - Red) / (NIR + Red), trong đó NIR là băng hồng ngoại gần (B08) và Red là băng phổ đỏ (B04).
ndvi = (data['B08'] - data['B04']) / (data['B08'] + data['B04']) # Đây vẫn là một Dask array, chưa được tính toán. Khi bạn gọi .compute(), Dask sẽ thực hiện tính toán trên tất cả các chunk và trả về kết quả cuối cùng.
result = ndvi.resample(time="1MS").mean().compute() # Tính giá trị trung bình của NDVI trên toàn bộ tile theo tháng. Dask sẽ quản lý việc tính toán và sử dụng bộ nhớ một cách hiệu quả.
print(f"Shape của kết quả NDVI sau khi resample theo tháng: {result.shape}, dtype: {result.dtype}")

### 29.2.3. Xử lý song song nhiều tiles

Dask Delayed là cơ chế của Dask cho phép biến các hàm Python thông thường thành các tác vụ thực thi lười (lazy execution). Thay vì chạy ngay, các phép tính được xây dựng thành một đồ thị tác vụ (task graph) và chỉ được thực thi khi gọi `.compute()`, giúp tối ưu bộ nhớ và xử lý song song.

Đối với tính toán NDVI trên ảnh vệ tinh, Dask Delayed có thể được sử dụng để chia ảnh thành nhiều khối (tiles), tính NDVI cho từng khối một cách song song, sau đó tổng hợp kết quả cuối cùng. Cách tiếp cận này đặc biệt hiệu quả với dữ liệu raster kích thước lớn vượt quá dung lượng RAM.

In [ ]:
outpath = r"G:\My Drive\python\geocourse\data\outputs"
import rioxarray as rxr
@dask.delayed
def process_tile(bbox, tile_index):
    # Tìm kiếm các item trong catalog của MPC cho tile_bbox
    search = catalog.search(
        collections=["sentinel-2-l2a"],
        bbox=bbox,
        datetime="2023-01-01/2023-12-31",
        query={"eo:cloud_cover": {"lt": 10}}  # Chỉ lấy ảnh có độ che phủ mây < 10%
    )

    items = list(search.item_collection())
    if len(items) == 0:
        print(f"⚠️ Không tìm thấy item nào cho bbox: {bbox}")
        return None

    # Đọc dữ liệu từ các item vào xarray dataset với stac_load
    data = stac_load(
        items = items,
        bands = ["B04", "B08"],  # Chỉ lấy 2 băng phổ đỏ và hồng ngoại gần để tính NDVI sau này
        chunks = {"x": 512, "y": 512}, # Chia nhỏ dữ liệu thành các chunk để xử lý với Dask
        resolution=10,
        patch_url=pc.sign,
        bbox=bbox
    ) 

    if data is None:
        print(f"⚠️ Không thể tải dữ liệu cho bbox: {bbox}")
        return None

    # Tính toán NDVI từ 2 băng phổ đỏ (B04) và hồng ngoại gần (B08)
    ndvi = (data['B08'] - data['B04']) / (data['B08'] + data['B04']) 
    result = ndvi.resample(time="1MS").mean().compute()
    result.rio.to_raster(f"{outpath}/ndvi_tile_{tile_index}.tif", compression = "LZW", dtype="float32") # Lưu kết quả NDVI trung bình theo tháng cho tile này vào file GeoTIFF. Bạn có thể thay đổi đường dẫn và tên file theo ý muốn.
    return None 

In [ ]:
# Tạo một danh sách các tác vụ tính toán NDVI cho từng tile bbox. Mỗi tác vụ sẽ được đánh dấu bằng @dask.delayed để Dask có thể quản lý việc phân phối công việc trên cluster và sử dụng bộ nhớ một cách hiệu quả.
tasks = [process_tile(bbox, i) for i, bbox in enumerate(bbox_tiles)]

results = dask.compute(*tasks) # Kích hoạt tính toán cho tất cả các tác vụ. Dask sẽ quản lý việc phân phối công việc trên cluster và sử dụng bộ nhớ một cách hiệu quả để xử lý dữ liệu lớn. Kết quả sẽ là một danh sách các xarray DataArray chứa NDVI trung bình theo tháng cho từng tile bbox. Bạn có thể sau đó kết hợp chúng lại thành một dataset lớn hơn nếu cần thiết.
# Ngoài ra, bạn có thể loop qua từng bounding box và tính toán NDVI cho từng tile một cách tuần tự nếu dữ liệu không quá lớn để xử lý trên máy cá nhân. 

## Tóm tắt

Bạn đã hoàn thành Bài 29 và nắm vững cách **xử lý dữ liệu vệ tinh quy mô lớn từ Microsoft Planetary Computer bằng Dask và kỹ thuật chia tile**.

### Các khái niệm chính đã nắm vững:
- ✅ Phân biệt hai cách đọc dữ liệu Sentinel-2: **đọc toàn bộ AOI** (phù hợp vùng nhỏ) và **đọc theo tile** (phù hợp vùng lớn, tiết kiệm bộ nhớ)
- ✅ Dùng `mercantile` để chia AOI thành các **XYZ tiles** theo zoom level, kiểm soát kích thước từng tile
- ✅ Tính chỉ số **NDVI** và tổng hợp theo tháng với `resample(time="1MS").mean()` trên một tile đơn lẻ
- ✅ Viết hàm xử lý tile với **`@dask.delayed`** để xây dựng task graph và thực thi song song nhiều tile cùng lúc
- ✅ Lưu kết quả từng tile dưới dạng **GeoTIFF** với nén `LZW` qua `rio.to_raster()`

### Kỹ năng bạn có thể áp dụng:
- Chia bất kỳ AOI cấp tỉnh/quốc gia nào thành các tile XYZ và kiểm soát kích thước xử lý qua zoom level
- Xây dựng pipeline song song với `dask.delayed` + `dask.compute()` để xử lý nhiều tile độc lập cùng lúc
- Giảm thiểu nguy cơ tràn bộ nhớ (out of memory) khi làm việc với dữ liệu vệ tinh hàng trăm GB bằng cách kết hợp tile-based processing và Dask chunking
- Lưu kết quả raster theo tile và có thể ghép lại sau bằng `rioxarray.merge_arrays()` hoặc GDAL
- Áp dụng kiến thức này để xây dựng các pipeline tính toán NDVI, phân loại lớp phủ, hoặc phát hiện thay đổi theo thời gian cho toàn bộ lãnh thổ Việt Nam
